# INT8 Backward dX with Fused Dequant

Bot task id: `14_backward_dx_int8_fused`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/14_backward_dx_int8_fused.ipynb)

## Условие

Беквард по входу с фьюзом деквантизации весов. Даны:
- `dY` формы `(B, OUT)` — fp16 (row-major, contiguous), апстрим-градиент;
- `W_q` формы `(IN, OUT)` — int8 (row-major, contiguous), квантованные веса;
- `s_w` — fp32 масштабы: скаляр (per-tensor) или вектор `(OUT,)` (per-OUT-channel).

Нужно вычислить `dX` формы `(B, IN)` в fp16:

$$W_{deq}[:, o] = W_q[:, o] \cdot s_w[o] \ (\text{или} \cdot s_w), \qquad dX[b, i] = \sum_k dY[b, k] \cdot W_{deq}[i, k]$$

- накапливайте $dY \cdot W_{deq}^T$ в fp32;
- результат пишется как fp16, хранение и доступ — row-major.

**Требования**
- ядро грузит тайлы `dY` (fp16) и `W_q` (int8), аккумулирует в fp32, умножает на `s_w` внутри K-цикла или один раз после него (можно умножать загружаемый тайл весов; масштабы не должны применяться повторно после суммирования);
- маскировать хвосты по всем измерениям;
- `tl.load` и `tl.store`;
- `@triton.autotune` с не менее чем 3 конфигурациями;
- режимы `per_channel=True/False` (constexpr-флаг `PER_CHANNEL`).

Масштаб per-channel идёт по `OUT`, а это ось суммирования, поэтому вынести его за K-цикл нельзя: умножайте тайл весов.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `14_backward_dx_int8_fused.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 14_backward_dx_int8_fused.py
import torch
import triton
import triton.language as tl


@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 64}, num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 64}, num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 64}, num_warps=4, num_stages=2),
    ],
    key=["B", "IN", "OUT"],
)
@triton.jit
def _backward_dx_fused_kernel(dy_ptr, wq_ptr, w_scale_ptr, dx_ptr,
                              B, IN, OUT,
                              BLOCK_M: tl.constexpr,
                              BLOCK_N: tl.constexpr,
                              BLOCK_K: tl.constexpr,
                              PER_CHANNEL: tl.constexpr):
    # YOUR CODE HERE
    pass


def backward_dx_int8_fused(dy: torch.Tensor,
                           w_q: torch.Tensor,
                           w_scale: torch.Tensor,
                           *, per_channel: bool = True) -> torch.Tensor:
    """Вернуть dX = dY @ (dequant(W_q))^T, dtype fp16, shape (B, IN)."""
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "14_backward_dx_int8_fused.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

B, IN, OUT = 77, 300, 130
dy = torch.randn(B, OUT, device="cuda", dtype=torch.half)
w_q = torch.randint(-127, 128, (IN, OUT), device="cuda", dtype=torch.int8)
for per_channel, s_w in ((True, torch.rand(OUT, device="cuda") * 1e-2), (False, torch.tensor([1e-2], device="cuda"))):
    ref = dy.float() @ (w_q.float() * s_w).T
    dx = sol.backward_dx_int8_fused(dy, w_q, s_w, per_channel=per_channel)
    assert dx.dtype == torch.float16 and dx.shape == (B, IN), (dx.dtype, dx.shape)
    torch.testing.assert_close(dx, ref.half(), atol=1e-2, rtol=1e-2)
assert len(sol._backward_dx_fused_kernel.configs) >= 3, "the task asks for at least 3 autotune configs"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("14_backward_dx_int8_fused.py")